# Train Boosting

W&B required. Run bootstrap first.

## 0) Bootstrap (Kaggle / Colab)

Run once per session: clone or pull this repo, `cd` into it, install deps. Locally you can skip clone if you already opened the notebook inside the repo.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str) -> None:
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

if (repo_path / ".git").exists():
    sh(f'git -C "{repo_path}" pull --ff-only')
elif (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using existing repo at", repo_path)
    try:
        sh("git pull --ff-only")
    except subprocess.CalledProcessError:
        print("git pull skipped/failed — continue with local tree")
else:
    sh(f'git clone {REPO_URL} "{repo_path}"')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

req = repo_path / "requirements-kaggle.txt"
if req.exists():
    sh(f'pip install -q -r "{req}"')
else:
    sh("pip install -q -r " + str(repo_path / "requirements.txt"))
sh("pip install -q -e .")
print("bootstrap OK")


## 1) Prepare and select the Boosting code

The bootstrap above is unchanged. Before running, enable Internet, attach the `vnts-processed` dataset, and grant this notebook access to the `WANDB_API_KEY` Secret. Your W&B account needs write access to the team project. Start with the CPU baseline.

Push your completed code to `feat/gradient-boosting` first. The next cell selects that branch after bootstrap and before importing the pipeline. Restart the kernel if you already imported `vn_tsc` before updating the code. Locally, the cell only checks the branch and does not switch it.


In [ ]:
BRANCH = "feat/gradient-boosting"
is_kaggle = Path("/kaggle/working").exists()

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo_path), *args], text=True
    ).strip()

if any(name == "vn_tsc" or name.startswith("vn_tsc.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before selecting or updating pipeline code.")

if is_kaggle:
    if git("status", "--porcelain"):
        raise RuntimeError("The checkout has local changes. Save them before updating the code.")
    git("fetch", "origin", BRANCH)
    git("switch", BRANCH)
    git("pull", "--ff-only", "origin", BRANCH)

current_branch = git("branch", "--show-current")
if current_branch != BRANCH:
    raise RuntimeError(f"Current branch: {current_branch}; expected: {BRANCH}.")

commit = git("rev-parse", "HEAD")
print("Branch:", current_branch)
print("Commit:", commit)


## 2) Configure W&B

Read the API key from Kaggle Secrets or a local environment variable. Never store it in the config or notebook output. Each call to `fit()` creates a W&B run.


In [ ]:
if is_kaggle:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

if not os.environ.get("WANDB_API_KEY", "").strip():
    raise RuntimeError("Missing WANDB_API_KEY. Configure the Secret or environment variable before training.")

print("W&B key is available. The pipeline will connect when fit() is called.")


## 3) Dataset and experiment overrides
Tune model parameters and pipeline options while preserving the shared splits and feature definitions. PCA is not implemented in this baseline and must remain disabled. After editing overrides, rerun this cell before training.


In [ ]:
PROCESSED_ROOT = ""
DATASET_VERSION = ""
RUN_NAME = "baseline"

if PROCESSED_ROOT:
    processed_root = Path(PROCESSED_ROOT)
elif is_kaggle:
    candidates = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("features/F_train.npy")})
    if len(candidates) != 1:
        raise RuntimeError(f"Set PROCESSED_ROOT explicitly. Candidates: {candidates}")
    processed_root = candidates[0]
else:
    processed_root = repo_path / "data" / "processed"

from vn_tsc.config.resolve import resolve_config

OVERRIDES = {
    "data": {"processed_root": str(processed_root)},
    "outputs": {"root": str(base / "outputs")},
    "model": {
        "backend": "lightgbm",
        "num_leaves": 63,
        "learning_rate": 0.05,
        "n_estimators": 150,
        "subsample_freq": 1,
    },
    "features": {"pca": {"enabled": False}},
    "train": {"with_aug": True, "require_wandb": True},
    "experiment": {
        "git_commit": commit,
        "git_branch": current_branch,
        "dataset_version": DATASET_VERSION or "not_recorded",
    },
}
cfg = resolve_config(
    pipeline_yaml="configs/pipelines/boosting.yaml",
    shared_yaml="configs/shared.yaml",
    runtime_yaml="configs/runtime/kaggle_2xt4.yaml",  
    overrides=OVERRIDES,
)
print("Processed root:", processed_root)
print("Model:", cfg["model"])
if not DATASET_VERSION:
    print("Dataset version is not recorded. Enter it before formal experiments.")


## 4) Validate inputs before starting a run

Check train and validation only; test is not loaded here. Image arrays must exist because the shared loader reads the corresponding X/y pair. Memory mapping checks array shapes without loading all images into RAM.


In [ ]:
import json
import numpy as np

required = ["metadata.csv", "class_table.csv", "features/scaler.joblib", "features/feature_report.json"]
splits = ["train", "val"]
if cfg["train"]["with_aug"]:
    splits.append("train_aug")
for split in splits:
    required.extend([f"features/F_{split}.npy", f"images/X_{split}.npy", f"images/y_{split}.npy"])
missing = [str(processed_root / name) for name in required if not (processed_root / name).is_file()]
if missing:
    raise FileNotFoundError("Missing files:\n" + "\n".join(missing))

report = json.loads((processed_root / "features/feature_report.json").read_text(encoding="utf-8"))
expected_dim = report["layout"]["total"]
for split in splits:
    F = np.load(processed_root / "features" / f"F_{split}.npy", mmap_mode="r")
    X = np.load(processed_root / "images" / f"X_{split}.npy", mmap_mode="r")
    y = np.load(processed_root / "images" / f"y_{split}.npy", mmap_mode="r")
    if F.ndim != 2 or y.ndim != 1 or not len(y) or not (len(F) == len(X) == len(y)):
        raise ValueError(f"Invalid array shapes in {split}")
    if F.shape[1] != expected_dim:
        raise ValueError(f"Feature dimension does not match feature_report: {split}")
    print(split, "features:", F.shape, "labels:", y.shape)
    del F, X, y


## 5) Train the baseline

This cell creates a new output directory and W&B run. Validation loss controls early stopping; use validation Macro-F1 to compare configurations. The pipeline logs loss history and final validation metrics to W&B. The saved configuration includes the code commit and library versions.


In [ ]:
from uuid import uuid4
from importlib.metadata import version
from vn_tsc.runtime.run_dir import make_run_dir
from vn_tsc.utils.seed import set_seed
from vn_tsc.pipelines.boosting.train import BoostingPipeline
from IPython.display import display
import pandas as pd

cfg["experiment"]["library_versions"] = {
    name: version(name)
    for name in ("numpy", "scikit-learn", "lightgbm", "wandb", "joblib")
}
set_seed(int(cfg.get("seed", 42)))
run_dir = make_run_dir(cfg["outputs"]["root"], "boosting", f"{RUN_NAME}_{uuid4().hex[:8]}")
pipeline = BoostingPipeline(cfg, run_dir)
train_metrics = pipeline.fit()
print("Run directory:", run_dir)
display(pd.DataFrame([train_metrics], index=["validation"]))


## 6) Verify the reloaded model

Create a fresh pipeline from the saved configuration, call `evaluate()` on validation, and compare its predictions with those saved during training. This cell does not retrain the model or create another W&B run.


In [ ]:
from vn_tsc.utils.io import load_yaml

reload_cfg = load_yaml(run_dir / "resolved_config.yaml")
reload_cfg.setdefault("eval", {})["split"] = "val"
reloaded_pipeline = BoostingPipeline(reload_cfg, run_dir)
val_metrics = reloaded_pipeline.evaluate()

with np.load(run_dir / "val_predictions.npz") as before:
    with np.load(run_dir / "evaluation" / "val" / "predictions.npz") as after:
        np.testing.assert_array_equal(before["y_true"], after["y_true"])
        np.testing.assert_array_equal(before["y_pred"], after["y_pred"])

print("OK: reloaded model predictions match training-time predictions.")
display(pd.DataFrame([val_metrics], index=["reloaded_validation"]))


## 7) Inspect and save validation plots

Read loss curves from `history.json` and the confusion matrix produced by `evaluate()`. Rows represent true labels and columns represent predictions. Normalize each row to compare classes with different sample counts. Plots are saved locally in the run directory.


In [ ]:
import matplotlib.pyplot as plt
from vn_tsc.data.classes import ClassTable
from vn_tsc.utils.io import load_json

figures_dir = run_dir / "figures"
figures_dir.mkdir(exist_ok=True)
history = load_json(run_dir / "history.json")
fig, ax = plt.subplots(figsize=(9, 4))
for split, values in history.items():
    loss = values.get("multi_logloss")
    if loss:
        ax.plot(range(1, len(loss) + 1), loss, label=split)
ax.set(xlabel="Boosting round", ylabel="Multi-class log loss", title="Training history")
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / "loss.png", dpi=160)
plt.show()
plt.close(fig)

matrix = np.load(run_dir / "evaluation" / "val" / "confusion_matrix.npy")
table = ClassTable.load(run_dir / "checkpoints" / "class_table.csv")
normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(15, 13))
heatmap = ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(table)), table.labels, rotation=90, fontsize=6)
ax.set_yticks(range(len(table)), table.labels, fontsize=6)
ax.set(xlabel="Predicted", ylabel="True", title="Validation confusion matrix (row-normalized)")
fig.colorbar(heatmap, ax=ax)
fig.tight_layout()
fig.savefig(figures_dir / "val_confusion_matrix.png", dpi=160)
plt.show()
plt.close(fig)


## 8) Final test evaluation

Enable only after selecting the configuration using validation. This evaluates the model in the current `run_dir`; confirm that it is the selected run. Evaluation results are saved locally and are not automatically logged to W&B.


In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    final_cfg = load_yaml(run_dir / "resolved_config.yaml")
    final_cfg.setdefault("eval", {})["split"] = "test"
    test_metrics = BoostingPipeline(final_cfg, run_dir).evaluate()
    display(pd.DataFrame([test_metrics], index=["test"]))
else:
    print("Test skipped. Continue selecting the configuration using validation.")


## 9) Package and download results

Rebuild the archive to include plots and evaluation outputs.


In [ ]:
from vn_tsc.runtime.pack import zip_run_dir
from IPython.display import FileLink

archive = zip_run_dir(run_dir)
print("Archive:", archive)
print("Size (MB):", round(archive.stat().st_size / 1024**2, 2))
display(FileLink(os.path.relpath(archive, Path.cwd())))
